# HW3 – Gradient Boosting Models
**Kaggle Playground Series S6E4** — Irrigation Need Prediction  


In [1]:
# Importing libraries
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import LabelEncoder, MinMaxScaler
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.metrics import (
    balanced_accuracy_score,
    classification_report,
    roc_auc_score,
)
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier

In [2]:
# ── Load data ─────────────────────────────────────────────────────────────────
train = pd.read_csv('train.csv')
test  = pd.read_csv('test.csv')
sample_sub = pd.read_csv('sample_submission.csv')

print(f'Train shape : {train.shape}')
print(f'Test shape  : {test.shape}')
train.head()

Train shape : (630000, 21)
Test shape  : (270000, 20)


,id,Soil_Type,Soil_pH,Soil_Moisture,Organic_Carbon,Electrical_Conductivity,Temperature_C,Humidity,Rainfall_mm,Sunlight_Hours,...,Crop_Type,Crop_Growth_Stage,Season,Irrigation_Type,Water_Source,Field_Area_hectare,Mulching_Used,Previous_Irrigation_mm,Region,Irrigation_Need
0,0,Loamy,4.92,32.58,1.01,3.05,15.01,50.61,725.99,5.90,...,Sugarcane,Sowing,Zaid,Drip,Rainwater,0.82,No,112.16,East,Low
1,1,Clay,7.08,56.61,0.44,2.00,22.92,67.86,985.66,6.98,...,Wheat,Vegetative,Kharif,Rainfed,River,5.27,Yes,47.16,South,Low
2,2,Clay,5.69,27.71,0.81,2.83,26.97,92.22,2201.70,6.05,...,Rice,Vegetative,Kharif,Sprinkler,Reservoir,8.24,Yes,110.38,North,Low
3,3,Sandy,5.65,13.32,1.33,0.87,13.32,61.57,1357.33,9.12,...,Wheat,Flowering,Kharif,Canal,River,8.32,Yes,53.85,South,Medium
4,4,Clay,7.96,59.14,0.38,0.96,20.22,91.11,1538.20,6.95,...,Wheat,Sowing,Rabi,Canal,River,7.37,No,93.19,South,Low


In [3]:
# Prep & Split

# Preserve Kaggle test IDs for submission
test_ids = test['id'].copy()

train = train.drop(columns=['id'])
test  = test.drop(columns=['id'])

# Separate features and target
X = train.drop(columns=['Irrigation_Need'])
y = train['Irrigation_Need']

# Encode the multi-class target to integers
le = LabelEncoder()
y_encoded = le.fit_transform(y)
print(f'Classes: {le.classes_}')

# One-hot encode categoricals across train + test jointly to align column shapes
X_all = pd.concat([X, test], axis=0)
X_all_encoded = pd.get_dummies(X_all, drop_first=True)

X_encoded = X_all_encoded.iloc[:len(X)].reset_index(drop=True)
test_encoded = X_all_encoded.iloc[len(X):].reset_index(drop=True)

# Validation split – stratified to preserve class distribution
X_train, X_val, y_train, y_val = train_test_split(
    X_encoded, y_encoded,
    test_size=0.2,
    random_state=42,
    stratify=y_encoded,
)

# Scale continuous features
scaler = MinMaxScaler()
X_train_s = scaler.fit_transform(X_train)
X_val_s = scaler.transform(X_val)
test_s = scaler.transform(test_encoded)

# Sample weights to handle class imbalance
sample_weights = compute_sample_weight(class_weight='balanced', y=y_train)

print(f'X_train : {X_train_s.shape}  |  X_val : {X_val_s.shape}  |  test : {test_s.shape}')

Classes: ['High' 'Low' 'Medium']
X_train : (504000, 35)  |  X_val : (126000, 35)  |  test : (270000, 35)


---
## Model 1 – XGBoost
Explore ≥3 different hyperparameter configurations below.

In [ ]:
# Initializing base XGB classifier model and determining baseline performance
xgb = XGBClassifier(random_state=42, n_jobs=1)

# Fitting xgb to the scaled training data with the sample weights
xgb.fit(X_train_s, y_train, sample_weight=sample_weights)

# Making initial predictions with the model
xgb_pred = xgb.predict(X_val_s)

# Finding initial accuracy
print(classification_report(xgb_pred, y_val))


              precision    recall  f1-score   support

           0       0.95      0.92      0.93      4343
           1       1.00      0.99      0.99     74605
           2       0.97      0.99      0.98     47052

    accuracy                           0.99    126000
   macro avg       0.97      0.96      0.97    126000
weighted avg       0.99      0.99      0.99    126000



In [ ]:
# Hyperparam config 1: GridSearchCV

param_grid = {
    ''
}

---
## Model 2 – LightGBM
Explore ≥3 different hyperparameter configurations below.

In [ ]:
# Your LightGBM experiments here


---
## (Optional) Model 3 – CatBoost

In [ ]:
# Your CatBoost experiments here


---
## Kaggle Submission

In [ ]:
# Retrain best model on full training data and generate submission
# X_full_s = scaler.fit_transform(X_encoded)
# full_weights = compute_sample_weight(class_weight='balanced', y=y_encoded)

# best_model.fit(X_full_s, y_encoded, sample_weight=full_weights)
# test_preds = best_model.predict(test_s)

# sub = pd.DataFrame({'id': test_ids, 'Irrigation_Need': le.inverse_transform(test_preds)})
# sub.to_csv('submission_hw3.csv', index=False)
# print('Submission saved ✓')